# 05 Change Mask Generation

Compare baseline and target glacier masks to produce four-class change labels and previews.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import rasterio
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from igcd.change_detection import generate_change_mask, summarize_change_mask
from igcd.config import load_config
from igcd.io import write_single_band_raster
from igcd.visualization import save_change_preview

config = load_config(PROJECT_ROOT / 'config' / 'config.json')
base_year = config.baseline_year
target_year = config.target_year
mask_root = config.paths['processed'] / 'masks'
change_root = config.paths['processed'] / 'change_masks'
preview_root = config.paths['reports'] / 'change_previews'
stats = []

for base_path in tqdm(sorted((mask_root / str(base_year)).glob('*_mask.tif'))):
    glacier_id = '_'.join(base_path.stem.split('_')[:2])
    target_path = mask_root / str(target_year) / f'{glacier_id}_{target_year}_mask.tif'
    if not target_path.exists():
        continue
    with rasterio.open(base_path) as src:
        baseline = src.read(1)
        profile = src.profile
        pixel_area = abs(src.transform.a * src.transform.e)
    with rasterio.open(target_path) as src:
        target = src.read(1)
    change = generate_change_mask(baseline, target)
    output = change_root / f'{glacier_id}_{base_year}_{target_year}_change.tif'
    write_single_band_raster(change, profile, output)
    save_change_preview(change, preview_root / f'{glacier_id}_change.png')
    summary = summarize_change_mask(change, pixel_area).__dict__
    summary.update({'glacier_id': glacier_id, 'change_mask': str(output)})
    stats.append(summary)

stats_df = pd.DataFrame(stats)
stats_df.to_csv(config.paths['reports'] / 'change_statistics.csv', index=False)
stats_df.head()